# ChemBreak 16 — Hierarchical Adaptive MDP with Frozen Holdout

**Environment:** Google Cloud Notebook Enterprise  
**Target:** ChemDFM  
**Training panel:** 24 tasks drawn only from the fixed CB12 **Train** partition  
**Unseen holdout:** 12 tasks drawn only from fixed CB12 **Test1**  
**Pipeline:** Train Baseline → 3 Learning Epochs → Freeze → Train-Panel Diagnostic → Holdout Baseline → Holdout Optimized → Results

CB16 fixes the CB15 state-sparsity problem by separating global behavioral knowledge, taxonomy-context knowledge, and lightweight task memory. The Test1 holdout is inaccessible until the policy is frozen. Run from Cell 1 downward.


In [ ]:
from pathlib import Path
import importlib, json, os, shutil, site, subprocess, sys

if 'runner' in globals():
    try: runner.close()
    except Exception: pass
    del runner

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
PROJECT_ID          = "rs-foundsecft-mghasemi"
REPO_URL            = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH              = "main"
PROJECT_SUBDIR      = "chembreak16"
EXPERIMENT_REVISION = "CB16_HIER_MDP_TRAIN24_TEST12_V1"
LIVE                = True
LIVE_PROGRESS       = True

content_root=Path('/content').resolve()
assert content_root.is_dir(), '/content unavailable — use Google Cloud Notebook Enterprise.'
if LIVE:
    assert PROJECT_ID.strip() and not PROJECT_ID.startswith('REPLACE_'), 'Set PROJECT_ID before live execution.'
storage_root=content_root/'chembreak16_storage'
model_cache=storage_root/'cache/huggingface/hub'
package_dir=storage_root/'python_packages'
env_paths={
    'HF_HOME':storage_root/'cache/huggingface',
    'HF_HUB_CACHE':model_cache,
    'HF_MODULES_CACHE':storage_root/'cache/huggingface/modules',
    'XDG_CACHE_HOME':storage_root/'cache/xdg',
    'TORCH_HOME':storage_root/'cache/torch',
    'TORCHINDUCTOR_CACHE_DIR':storage_root/'cache/torchinductor',
    'TRITON_CACHE_DIR':storage_root/'cache/triton',
    'CUDA_CACHE_PATH':storage_root/'cache/cuda',
    'PIP_CACHE_DIR':storage_root/'cache/pip',
    'TMPDIR':storage_root/'tmp',
}
for key,path in env_paths.items():
    path.mkdir(parents=True,exist_ok=True)
    os.environ[key]=str(path)
os.environ['TMP']=os.environ['TMPDIR']
os.environ['TEMP']=os.environ['TMPDIR']
print('Storage:',storage_root)


## C1 — Clone or update the GitHub repository

The notebook is the controller. The CB16 source comes from the `chembreak16` folder in GitHub.


In [ ]:
checkout=content_root/'chembreak16_repo'
def git(*args,cwd=None):
    subprocess.run(['git',*args],cwd=cwd,check=True)
if not (checkout/'.git').is_dir():
    git('clone','--branch',BRANCH,'--single-branch',REPO_URL,str(checkout))
else:
    git('fetch','origin',BRANCH,cwd=checkout)
    git('checkout',BRANCH,cwd=checkout)
    git('pull','--ff-only','origin',BRANCH,cwd=checkout)
PROJECT_DIR=(checkout/PROJECT_SUBDIR).resolve()
assert (PROJECT_DIR/'pyproject.toml').is_file(), f'CB16 package not found at {PROJECT_DIR}. Push the chembreak16 folder to GitHub first.'
os.chdir(PROJECT_DIR)
print('Project:',PROJECT_DIR)


## C2 — Verify Train/Test firewall and fixed selections

CB16 uses the already-fixed CB12 partition. Training uses only 24 tasks from **Train**. The unseen 12-task generalization panel comes only from **Test1**. The sets are locked and disjoint.


In [ ]:
sys.path.insert(0,str(PROJECT_DIR/'src'))
for module_name in [name for name in list(sys.modules) if name=='chembreak16' or name.startswith('chembreak16.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()
from chembreak16.selection import verify_bundle
report=verify_bundle(
    PROJECT_DIR/'data/final_task_bank.csv',
    PROJECT_DIR/'data/CB12_partition_manifest_v1.csv',
    PROJECT_DIR/'data/CB12_partition_lock_v1.json',
    PROJECT_DIR/'data/CB16_train24_manifest_v1.csv',
    PROJECT_DIR/'data/CB16_holdout12_manifest_v1.csv',
    PROJECT_DIR/'data/CB16_selection_lock_v1.json',
)
print(json.dumps(report,indent=2,sort_keys=True))


## C3 — Install the CB16 dependency stack

Dependencies and model caches stay under `/content/chembreak16_storage`. CB16 does not reuse CB15 policy/checkpoint state.


In [ ]:
compatibility_specs=['transformers==4.40.2','tokenizers==0.19.1','huggingface-hub==0.23.5','safetensors==0.4.5','accelerate==0.30.1','sentencepiece==0.2.0','einops==0.8.1']
marker=package_dir/'cb16_compatibility.json'
expected={'specifications':compatibility_specs}
installed=json.loads(marker.read_text()) if marker.exists() else None
if installed!=expected:
    print('Installing CB16 compatibility stack (first run only)...')
    subprocess.run([sys.executable,'-m','pip','install','--target',str(package_dir),'--cache-dir',str(env_paths['PIP_CACHE_DIR']),'--no-deps','--upgrade',*compatibility_specs],check=True)
    marker.write_text(json.dumps(expected,indent=2))
else:
    print('Compatibility stack already installed.')
subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(package_dir),'--cache-dir',str(env_paths['PIP_CACHE_DIR']),'google-auth>=2.35,<3','google-genai>=1.47,<2','pandas>=2.2,<3','numpy>=1.26,<3','PyYAML>=6,<7'],check=True)
site.addsitedir(str(package_dir))
sys.path.insert(0,str(package_dir))
sys.path.insert(0,str(PROJECT_DIR/'src'))
importlib.invalidate_caches()
expected_versions={'transformers':'4.40.2','tokenizers':'0.19.1','huggingface_hub':'0.23.5','accelerate':'0.30.1'}
for name,expected_version in expected_versions.items():
    mod=importlib.import_module(name)
    path=Path(mod.__file__).resolve()
    actual=str(getattr(mod,'__version__','?'))
    low=str(path).lower()
    if 'chembreak' in low and 'chembreak16_storage' not in low:
        raise RuntimeError(f'{name} loaded from another ChemBreak storage: {path}. Restart kernel.')
    if actual!=expected_version:
        raise RuntimeError(f'{name} version mismatch: {actual} != {expected_version}')
print('Compatibility imports verified.')


## C4 — Build runtime configuration

This sets environment-specific paths only. It does not regenerate or alter Train/Test membership.


In [ ]:
import yaml
base=yaml.safe_load((PROJECT_DIR/'configs/config.cb16.yaml').read_text())
base['run'].update({
    'project_root':str(PROJECT_DIR),
    'task_bank_path':str(PROJECT_DIR/'data/final_task_bank.csv'),
    'partition_manifest_path':str(PROJECT_DIR/'data/CB12_partition_manifest_v1.csv'),
    'partition_lock_path':str(PROJECT_DIR/'data/CB12_partition_lock_v1.json'),
    'train_manifest_path':str(PROJECT_DIR/'data/CB16_train24_manifest_v1.csv'),
    'holdout_manifest_path':str(PROJECT_DIR/'data/CB16_holdout12_manifest_v1.csv'),
    'selection_lock_path':str(PROJECT_DIR/'data/CB16_selection_lock_v1.json'),
    'output_root':str(storage_root/'runs'),
    'dry_run':not LIVE,
    'experiment_revision':EXPERIMENT_REVISION,
    'train_task_limit':None,
    'holdout_task_limit':None,
    'live_progress':LIVE_PROGRESS,
})
base['targets'][0]['cache_dir']=str(model_cache)
base['targets'][0]['offload_folder']=str(storage_root/'offload/ChemDFM')
policy_dir=storage_root/'policies'/EXPERIMENT_REVISION
base['policy']['training_artifact_path']=str(policy_dir/'training_policy.json')
base['policy']['frozen_artifact_path']=str(policy_dir/'frozen_policy.json')
runtime_path=storage_root/f'runtime_{EXPERIMENT_REVISION}.yaml'
runtime_path.write_text(yaml.safe_dump(base,sort_keys=False))
if LIVE:
    os.environ['GOOGLE_CLOUD_PROJECT']=PROJECT_ID
    os.environ['CHEMBREAK_ENABLE_LIVE']='YES'
print('Runtime config:',runtime_path)
print('LIVE:',LIVE,'| LIVE_PROGRESS:',LIVE_PROGRESS,'| Project:',os.environ.get('GOOGLE_CLOUD_PROJECT','mock'))


## C5 — Preflight

Checks the frozen CB12 partition hashes, Train24/Test1-Holdout12 lock, zero overlap, CUDA/tokenizer compatibility, and Vertex structured-output probes.


In [ ]:
from chembreak16.preflight import run_preflight
preflight=run_preflight(runtime_path,probe_tokenizer=LIVE,probe_roles=LIVE)
print(json.dumps(preflight,indent=2,sort_keys=True))
assert preflight['status']=='ok'


## C6 — Create runner and load ChemDFM once

Maximum planned episodes: **144**. Maximum target queries: **468**; usually lower because successful episodes stop early.


In [ ]:
from chembreak16.runner import ChemBreak16Runner
runner=ChemBreak16Runner(runtime_path)
runner.load_target()
print('Runner ready.')
print('Planned episodes: 24 train baseline + 72 learning + 24 train optimized + 12 holdout baseline + 12 holdout optimized = 144')
print('Maximum target queries: 468')


## C7 — Phase 1: Train baseline

Only the fixed Train24 panel is touched. With `LIVE_PROGRESS=True`, each judged result and running ASR appears immediately.

The per-episode completion line includes `running_ASR` for the active phase.


In [ ]:
train_baseline_summary=runner.run_train_baseline()
print(json.dumps(train_baseline_summary,indent=2,sort_keys=True))


## C8 — Phase 2: Learning on Train24 only

Base epsilon remains **0.30 → 0.20 → 0.15**. CB16 now prints hierarchical diagnostics: global Q (`Qg`), HC/HD/OT context Qs (`Qhc`, `Qhd`, `Qot`), task Q (`Qt`), active knowledge components, support visits, repetition penalty, and blocked actions.

`mode=cold_start` means the policy has no learned support for that state; it is no longer mislabeled as exploitation.


In [ ]:
learning_summary=runner.run_learning()
print(json.dumps(learning_summary,indent=2,sort_keys=True))


## C9 — Freeze policy

The Test1 holdout remains inaccessible until this succeeds. After freezing, no Q update is permitted during either optimized evaluation.


In [ ]:
frozen=runner.freeze_policy()
print(json.dumps(frozen,indent=2,sort_keys=True))


## C10 — Phase 3: Train-panel optimized diagnostic

This measures within-panel improvement only. It is **not** the generalization result.


In [ ]:
train_optimized_summary=runner.run_train_optimized()
print(json.dumps(train_optimized_summary,indent=2,sort_keys=True))


## C11 — Phase 4: Unseen Test1 holdout baseline

This is the first time the 12 holdout tasks are accessed. The policy is already frozen, so these responses cannot influence learning.


In [ ]:
holdout_baseline_summary=runner.run_holdout_baseline()
print(json.dumps(holdout_baseline_summary,indent=2,sort_keys=True))


## C12 — Phase 5: Unseen Test1 holdout optimized evaluation

Frozen policy, `epsilon=0`, and **no Q updates**. Task-specific memory is unavailable for these unseen assignment IDs, so performance must come from global and taxonomy-context transfer plus the fixed decision controls.


In [ ]:
holdout_optimized_summary=runner.run_holdout_optimized()
print(json.dumps(holdout_optimized_summary,indent=2,sort_keys=True))


## C13 — Results

The headline generalization comparison is **Holdout Baseline ASR vs Holdout Optimized ASR**. `summary.json` reports `null` rather than a false zero for any phase that has not run.

Detailed per-decision values are also exported to `policy_diagnostics.csv`.


In [ ]:
summary=runner.export_results()
print(json.dumps(summary,indent=2,sort_keys=True))
release_dir=storage_root/'runs'/EXPERIMENT_REVISION/'release'
print('Release directory:',release_dir)
print('Files:',[p.name for p in sorted(release_dir.iterdir())])


## C14 — Build results ZIP and close model

The ZIP contains result tables, checkpoint, runtime configuration, CB12 partition provenance, CB16 Train/Holdout locks, and both policy artifacts. Model weights/caches are excluded.


In [ ]:
import zipfile
runner.close()
run_dir=storage_root/'runs'/EXPERIMENT_REVISION
release_dir=run_dir/'release'
zip_path=content_root/f'{EXPERIMENT_REVISION}_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in release_dir.rglob('*'):
        if p.is_file():
            z.write(p,p.relative_to(run_dir))
    checkpoint=run_dir/'state.sqlite3'
    if checkpoint.exists():
        z.write(checkpoint,Path('checkpoint')/checkpoint.name)
    if runtime_path.exists():
        z.write(runtime_path,Path('provenance')/runtime_path.name)
    for p in [
        PROJECT_DIR/'data/CB12_partition_manifest_v1.csv',
        PROJECT_DIR/'data/CB12_partition_lock_v1.json',
        PROJECT_DIR/'data/CB16_train24_manifest_v1.csv',
        PROJECT_DIR/'data/CB16_holdout12_manifest_v1.csv',
        PROJECT_DIR/'data/CB16_selection_lock_v1.json',
    ]:
        z.write(p,Path('provenance')/p.name)
    for p in [policy_dir/'training_policy.json',policy_dir/'frozen_policy.json']:
        if p.exists():
            z.write(p,Path('policies')/p.name)
print('Results ZIP:',zip_path)
